# 02 — Sliding Window and Prefix Sum

## Why This Matters for DSA
Sliding window is same-direction two pointers with one more idea layered on: instead of a `write` pointer tracking "what to keep," both pointers together bound a contiguous *range*, and you maintain some running state (a sum, a character count, a max) as that range grows and shrinks. Prefix sum solves an adjacent but different problem: answering MANY range-sum queries fast, including cases sliding window structurally can't handle (negative numbers). Between the two, you'll cover the large majority of "subarray" and "substring" problems you'll ever meet.

## Prerequisites
- `01_Arrays_and_Two_Pointers` — this notebook's sliding window sections are a direct extension of Section 3 (same-direction pointers) from that notebook

## Learning Objectives
By the end of this notebook, you will be able to:
- Apply fixed-size sliding window to turn an O(n·k) brute force into O(n)
- Apply variable-size sliding window with both the "expand until invalid, shrink" and "expand until valid, shrink while still valid" shapes
- Explain why a variable window's nested-loop APPEARANCE is still O(n) overall
- Build a 1D prefix sum array and use it for O(1) range-sum queries after O(n) preprocessing
- Combine prefix sums with a hash map to count subarrays matching a target sum in one pass, including with negative numbers
- Extend prefix sums to 2D for O(1) rectangular range queries on a grid
- Recognize when a problem needs sliding window versus when it needs prefix sum instead — they look similar but solve different underlying constraints

## Checklist Before Moving On
- [ ] I can implement fixed-size sliding window without recomputing the window sum from scratch each slide
- [ ] I can implement both shapes of variable-size window and explain when each applies
- [ ] I can justify why a variable window with a nested while loop is still O(n), not O(n²)
- [ ] I can derive the range-sum formula `prefix[right+1] - prefix[left]` without memorizing it blindly
- [ ] I know why negative numbers break sliding window's shrink logic but don't break prefix sum + hash map
- [ ] I can extend the 1D prefix sum inclusion-exclusion idea to 2D

## Table of Contents
1. Fixed-Size Sliding Window
2. Variable-Size Sliding Window — Expand and Track (Longest Valid Window)
3. Variable-Size Sliding Window — Expand Then Shrink (Smallest Valid Window)
4. Prefix Sum — O(1) Range Queries After O(n) Preprocessing
5. Prefix Sum + Hash Map — Counting Subarrays With a Target Sum
6. 2D Prefix Sum
7. Sliding Window vs Prefix Sum — Choosing the Right Tool
8. Phase Checkpoint, Cheat Sheet, and Answer Key
9. LeetCode Practice Problems


## 1. Fixed-Size Sliding Window

### The Why
This is the simplest window shape and the clearest illustration of the core sliding-window idea: consecutive windows overlap heavily, so recomputing each one from scratch wastes almost all of that overlap. Sliding the window instead reuses everything except the one element entering and the one element leaving.

### Core Mechanism
- Build the FIRST window directly (there's no previous window to slide from yet).
- For every subsequent position, update the running value in O(1): add the element newly entering the window, subtract the element that just left it (the element `k` positions behind the current right edge).
- **Why this beats brute force:** recomputing every window's sum from scratch is O(n·k) — for each of the roughly `n` windows, re-summing all `k` elements. Sliding does O(1) work per position, giving O(n) total.

### Common Pitfalls
- Recomputing the window sum from scratch on every slide "to be safe" — this silently reintroduces the O(n·k) cost the pattern exists to eliminate; the whole point is the O(1) incremental update.
- Off-by-one errors on which index is "leaving" the window — the element leaving is at `i - k`, not `i - k + 1` or `i - k - 1`; double check against a small hand-traced example if unsure.


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
using namespace std;

// FIXED-SIZE SLIDING WINDOW: the window size k never changes -- you slide it one
// position at a time, and at each slide you ADD the new element entering the window
// and REMOVE the element leaving it, instead of recomputing the whole window's sum.
int maxSumSubarray(vector<int>& nums, int k) {
    int n = (int)nums.size();

    // build the FIRST window the naive way -- there's no "previous window" to slide from yet
    int windowSum = 0;
    for (int i = 0; i < k; i++) windowSum += nums[i];

    int best = windowSum;

    // slide the window forward one element at a time
    for (int i = k; i < n; i++) {
        windowSum += nums[i];        // add the new element entering the window
        windowSum -= nums[i - k];    // remove the element leaving the window (k positions back)
        best = max(best, windowSum);
    }

    return best;
}

int main() {
    vector<int> nums{2, 1, 5, 1, 3, 2};
    cout << "max sum of window size 3: " << maxSumSubarray(nums, 3) << "\n";   // expected 9 (5+1+3)

    // WHY THIS BEATS BRUTE FORCE: recomputing each window's sum from scratch is O(n*k) --
    // for every one of the ~n windows, you re-sum all k elements. Sliding the window instead
    // does O(1) work per slide (one add, one subtract), giving O(n) total -- the key insight
    // is that consecutive windows overlap in k-1 elements, so almost all the work is reusable

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 2. Variable-Size Sliding Window — Expand and Track (Longest Valid Window)

### The Why
Many problems don't have a fixed window size at all — the goal IS to find the best window size, subject to some validity condition. This is same-direction two pointers from the previous notebook, with a twist: instead of a `write` pointer that only moves when keeping something, `left` moves specifically to RESTORE validity when the window becomes invalid.

### Core Mechanism
- `right` always expands forward, one step at a time, unconditionally.
- Track whatever state defines validity — here, the most recent index each character was seen at.
- When expanding `right` would make the window invalid (a repeated character INSIDE the current window), jump `left` forward to just past that repeat's previous occurrence — restoring validity in one step rather than incrementing `left` one position at a time.
- At every position, record the current window size as a candidate for the best answer.
- **Why this is still O(n), despite looking like it could be worse:** `right` visits every index exactly once. `left` ALSO only ever moves forward — never backward. Two pointers that individually only move forward, no matter how they're interleaved, do a total of at most O(n) combined movement across the entire run.

### Common Pitfalls
- Checking `lastSeenIndex.count(c)` alone without also checking `lastSeenIndex[c] >= left` — a character seen before but OUTSIDE the current window (already shrunk past) isn't actually a conflict, and incorrectly jumping `left` backward-in-effect (or failing to jump forward when needed) breaks the algorithm.


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <string>
#include <unordered_map>
using namespace std;

// VARIABLE-SIZE SLIDING WINDOW: the window grows and shrinks based on a CONDITION,
// not a fixed size. Two pointers (left, right) bound the window; right always expands
// forward, left only moves forward when the current window becomes invalid.
int lengthOfLongestSubstring(string s) {
    unordered_map<char, int> lastSeenIndex;   // character -> most recent index it appeared at
    int left = 0, best = 0;

    for (int right = 0; right < (int)s.size(); right++) {
        char c = s[right];

        // if c was seen before AND that occurrence is still inside the current window,
        // shrink the window by jumping `left` past that previous occurrence
        if (lastSeenIndex.count(c) && lastSeenIndex[c] >= left) {
            left = lastSeenIndex[c] + 1;
        }

        lastSeenIndex[c] = right;
        best = max(best, right - left + 1);   // window size = right - left + 1
    }

    return best;
}

int main() {
    cout << lengthOfLongestSubstring("abcabcbb") << "\n";   // expected 3 ("abc")
    cout << lengthOfLongestSubstring("bbbbb") << "\n";      // expected 1 ("b")
    cout << lengthOfLongestSubstring("pwwkew") << "\n";     // expected 3 ("wke")

    // WHY THIS IS O(n), NOT O(n^2): `right` visits every index exactly once, moving forward
    // only. `left` also only ever moves FORWARD, never backward -- across the whole run,
    // left's total movement is bounded by n. Two pointers that each only move forward,
    // combined, do at most O(n) total work, even though it LOOKS like a nested-loop shape
    // (a window inside a scan) at first glance

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 3. Variable-Size Sliding Window — Expand Then Shrink (Smallest Valid Window)

### The Why
This is the other common shape of variable window, used when you want the SMALLEST window satisfying a condition rather than the longest. The structure looks similar to Section 2 but the shrink logic is a `while` loop rather than a single conditional jump — worth knowing both shapes, since which one a problem needs isn't always obvious from the description alone.

### Core Mechanism
- `right` expands the window, unconditionally, adding to the running sum.
- Once the window satisfies the condition (`windowSum >= target`), shrink from the left AS MUCH AS POSSIBLE while it's still valid — recording the window size at every point it remains valid, since shrinking further might still work and give an even smaller answer.
- This greedily finds, for each `right`, the smallest valid window ENDING at that position — and the overall answer is the best across all of them.
- **Still O(n) overall, same argument as Section 2:** even though there's a `while` loop nested inside the `for` loop, `left`'s total movement across the WHOLE run (not per outer iteration) is bounded by `n`, since it only ever increases and can't exceed the array length.

### Common Pitfalls
- Treating the nested `while` loop as automatic evidence of O(n²) — always check whether the inner loop's total work across the FULL run is bounded, not just whether it's nested. This is the same reasoning as the loop-analysis rules in `01_Complexity_Analysis`, applied to a less obvious shape.
- Forgetting to handle the "no valid window exists" case (returning a sentinel like 0, as shown, when the target sum can never be reached).


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
#include <climits>
using namespace std;

// The other common shape of variable window: EXPAND until a condition is satisfied,
// then SHRINK as much as possible while it's still satisfied, recording the best
// (usually smallest) window at each point it's valid.
int minSubArrayLen(int target, vector<int>& nums) {
    int left = 0, windowSum = 0;
    int best = INT_MAX;

    for (int right = 0; right < (int)nums.size(); right++) {
        windowSum += nums[right];              // expand: always add the new right element

        // shrink from the left AS MUCH AS POSSIBLE while the window still meets the condition --
        // this greedily finds the SMALLEST valid window ending at the current `right`
        while (windowSum >= target) {
            best = min(best, right - left + 1);
            windowSum -= nums[left];
            left++;
        }
    }

    return best == INT_MAX ? 0 : best;   // 0 signals "no valid window exists"
}

int main() {
    vector<int> nums{2, 3, 1, 2, 4, 3};
    cout << minSubArrayLen(7, nums) << "\n";   // expected 2 (subarray [4,3])

    vector<int> nums2{1, 4, 4};
    cout << minSubArrayLen(4, nums2) << "\n";  // expected 1 (subarray [4])

    vector<int> nums3{1, 1, 1, 1};
    cout << minSubArrayLen(11, nums3) << "\n"; // expected 0 -- no subarray sums to 11

    // this is STILL O(n) overall: even though there's a `while` loop nested inside the `for`
    // loop, `left` only ever moves forward, and its total movement across the ENTIRE run
    // is bounded by n (it can advance at most n times total, not n times PER outer iteration) --
    // this is the same "total forward movement is bounded" argument as Section 1's variable window

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 4. Prefix Sum — O(1) Range Queries After O(n) Preprocessing

### The Why
Sliding window is great for finding an optimal (longest/shortest) window, but it's the wrong tool when you need to answer MANY range-sum queries on a FIXED array — prefix sum is built exactly for that: pay a one-time O(n) cost, then answer any range-sum query in O(1).

### Core Mechanism
- `prefix[i]` stores the sum of everything BEFORE index `i` (i.e. `nums[0..i-1]`) — note the deliberate off-by-one shift: `prefix[0] = 0` represents "the sum of nothing," which lets range queries starting at index 0 use the exact same formula as any other range, with no special case.
- Building it is a simple running total: `prefix[i+1] = prefix[i] + nums[i]`.
- **The range-sum formula:** `rangeSum(left, right) = prefix[right+1] - prefix[left]`. Intuition: `prefix[right+1]` is the sum of everything up to and including `right`; `prefix[left]` is the sum of everything strictly before `left`; subtracting removes exactly the unwanted prefix, leaving exactly the range `[left, right]`.
- With `Q` queries against the same array, this is O(n + Q) total — versus O(n·Q) if you summed each range fresh every time. The more queries you have against the same static array, the bigger this win gets.

### Common Pitfalls
- Forgetting the `+1` shift and using `prefix[right] - prefix[left]` directly — this quietly excludes one endpoint of the intended range. Always re-derive the formula from what `prefix[i]` actually means rather than guessing at index arithmetic.
- Using prefix sum on an array that changes frequently — every update would require rebuilding the affected suffix of the prefix array, which is expensive if updates are common; for a MUTABLE array with frequent updates and range queries, a Fenwick Tree or Segment Tree (see the README's Advanced Trees section) is the right tool instead.


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
using namespace std;

// PREFIX SUM: precompute cumulative sums ONCE in O(n), then answer ANY range-sum query
// in O(1) -- versus summing the range fresh every time, which is O(n) PER query.
class PrefixSum {
    vector<long long> prefix;   // prefix[i] = sum of nums[0..i-1] (note the shift by 1)
public:
    PrefixSum(vector<int>& nums) {
        prefix.resize(nums.size() + 1, 0);
        for (int i = 0; i < (int)nums.size(); i++) {
            prefix[i + 1] = prefix[i] + nums[i];   // running total
        }
        // WHY THE +1 SHIFT: prefix[0] = 0 (sum of nothing) lets range queries starting
        // at index 0 be handled with the SAME formula as any other range, no special case
    }

    // sum of nums[left..right] inclusive
    long long rangeSum(int left, int right) {
        return prefix[right + 1] - prefix[left];
        // prefix[right+1] = sum of everything up to and including `right`
        // prefix[left]    = sum of everything BEFORE `left`
        // subtracting removes exactly the part before `left`, leaving [left, right]
    }
};

int main() {
    vector<int> nums{1, 2, 3, 4, 5, 6};
    PrefixSum ps(nums);

    cout << "sum[1..3] = " << ps.rangeSum(1, 3) << "\n";   // 2+3+4 = 9
    cout << "sum[0..5] = " << ps.rangeSum(0, 5) << "\n";   // whole array = 21
    cout << "sum[4..4] = " << ps.rangeSum(4, 4) << "\n";   // single element = 5

    // O(n) to build the prefix array once, then O(1) per query -- if you have Q queries,
    // this is O(n + Q) total, versus O(n*Q) recomputing each range sum from scratch --
    // a massive win whenever many queries hit the same underlying array

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 5. Prefix Sum + Hash Map — Counting Subarrays With a Target Sum

### The Why
This is the pattern that answers a question sliding window structurally CANNOT handle correctly: "how many subarrays sum to exactly k?" when the array contains negative numbers. Sliding window's shrink logic relies on a monotonic relationship (adding an element only increases the sum) — negative numbers break that assumption entirely.

### Core Mechanism
- **The algebra:** a subarray from index `i` to `j-1` sums to `target` exactly when `prefix[j] - prefix[i] == target`. Rearranged: `prefix[i] == prefix[j] - target`. So at each position `j`, you need to know how many EARLIER prefix sums equal `prefix[j] - target`.
- A hash map storing "prefix sum value → how many times it's occurred so far" answers that in O(1) per position, instead of checking every earlier index individually (which would be O(n) per position, O(n²) overall).
- **The `prefixCount[0] = 1` initialization matters:** it represents an "empty prefix" existing before the array starts, which correctly handles subarrays that begin at index 0 (their required earlier-prefix-sum is exactly 0, which needs to already be in the map before any real elements are processed).
- Update the map with the CURRENT prefix sum only AFTER checking for a match — this ensures a subarray never counts itself as its own match (that would require `target == 0` matching against itself trivially, which isn't a valid subarray of length 0 in this problem's definition).

### Common Pitfalls
- Trying to solve this with sliding window when negative numbers are possible — a window's sum doesn't change monotonically as it grows if the array can contain negatives, so the greedy shrink logic from Section 3 gives wrong answers. Prefix sum + hash map has no such requirement, which is exactly why it's the right tool here.
- Updating the hash map BEFORE checking for a match instead of after — this can incorrectly let a prefix sum match against itself in edge cases.


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
#include <unordered_map>
using namespace std;

// PREFIX SUM + HASH MAP: count subarrays summing to a target, in a SINGLE pass, O(n).
//
// Key algebra: if prefix[j] - prefix[i] == target, then the subarray from i to j-1 sums
// to target. Rearranged: prefix[i] == prefix[j] - target. So at each position j, we just
// need to know HOW MANY earlier prefix sums equal (prefix[j] - target) -- a hash map
// gives us that count in O(1), instead of checking every earlier index individually.
int subarraySum(vector<int>& nums, int k) {
    unordered_map<long long, int> prefixCount;
    prefixCount[0] = 1;   // an empty prefix (sum 0) exists "before" the array starts --
                           // this handles subarrays that start at index 0 correctly

    long long runningSum = 0;
    int count = 0;

    for (int num : nums) {
        runningSum += num;

        // how many earlier prefix sums equal (runningSum - k)? each one marks a valid
        // subarray ending at the current position
        if (prefixCount.count(runningSum - k)) {
            count += prefixCount[runningSum - k];
        }

        prefixCount[runningSum]++;   // record the CURRENT prefix sum for future positions to use
    }

    return count;
}

int main() {
    vector<int> nums{1, 1, 1};
    cout << subarraySum(nums, 2) << "\n";   // expected 2: [1,1] at (0,1) and (1,2)

    vector<int> nums2{1, 2, 3};
    cout << subarraySum(nums2, 3) << "\n";  // expected 2: [1,2] and [3]

    vector<int> nums3{1, -1, 0};
    cout << subarraySum(nums3, 0) << "\n";  // expected 3: [1,-1], [0], [1,-1,0] -- negatives
                                              // are exactly why this needs prefix sums + hash map
                                              // rather than a sliding window (window approaches
                                              // require monotonic behavior that negatives break)

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 6. 2D Prefix Sum

### The Why
The same idea as 1D prefix sum extends cleanly to a grid — useful whenever you need many rectangular-region sum queries on a static 2D array (image processing, grid-based DP precomputation, matrix range queries).

### Core Mechanism
- `prefix[r][c]` stores the sum of the rectangle from `(0,0)` to `(r-1, c-1)` — same off-by-one shift idea as 1D, extended to two dimensions.
- **Building it uses inclusion-exclusion:** `prefix[r+1][c+1] = prefix[r][c+1] + prefix[r+1][c] - prefix[r][c] + grid[r][c]`. Adding the cell above and the cell to the left double-counts the region diagonally above-left of the current cell, so that region gets subtracted once to correct for the double-count, before finally adding the current cell's own value.
- **The range-sum query uses the same inclusion-exclusion logic, in reverse:** take the big rectangle up to `(r2, c2)`, subtract everything above row `r1` and everything left of column `c1` — but this subtracts the top-left corner region TWICE (once in each subtraction), so it gets added back once to correct for that.
- This generalizes: whenever you're combining overlapping "up to here" regions, inclusion-exclusion (add the parts, subtract the double-counted overlap) is the pattern to reach for.

### Common Pitfalls
- Getting the inclusion-exclusion signs backwards during either the build step or the query step — if results are consistently off by exactly one row's or column's worth of sum, that's usually the signature of a sign error here. Re-derive from a small hand-drawn 2×2 or 3×3 grid example if a bug appears.


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
using namespace std;

// 2D PREFIX SUM: the same idea as 1D, extended to a grid -- precompute once in O(rows*cols),
// then answer any rectangular sum query in O(1).
class PrefixSum2D {
    vector<vector<long long>> prefix;   // prefix[r][c] = sum of everything in rows [0,r), cols [0,c)
public:
    PrefixSum2D(vector<vector<int>>& grid) {
        int rows = grid.size(), cols = grid[0].size();
        prefix.assign(rows + 1, vector<long long>(cols + 1, 0));

        for (int r = 0; r < rows; r++) {
            for (int c = 0; c < cols; c++) {
                // inclusion-exclusion: add the cell above, add the cell to the left,
                // subtract the cell diagonally above-left (it got counted in BOTH additions,
                // so subtract it once to correct for the double-count), then add the current cell
                prefix[r+1][c+1] = prefix[r][c+1] + prefix[r+1][c] - prefix[r][c] + grid[r][c];
            }
        }
    }

    // sum of the rectangle with corners (r1,c1) top-left and (r2,c2) bottom-right, inclusive
    long long rangeSum(int r1, int c1, int r2, int c2) {
        return prefix[r2+1][c2+1] - prefix[r1][c2+1] - prefix[r2+1][c1] + prefix[r1][c1];
        // same inclusion-exclusion logic in reverse: take the big rectangle up to (r2,c2),
        // remove everything above row r1 and everything left of column c1, then add back
        // the top-left corner region that got subtracted TWICE
    }
};

int main() {
    vector<vector<int>> grid{
        {3, 0, 1, 4, 2},
        {5, 6, 3, 2, 1},
        {1, 2, 0, 1, 5},
        {4, 1, 0, 1, 7},
        {1, 0, 3, 0, 5}
    };
    PrefixSum2D ps(grid);

    cout << ps.rangeSum(2, 1, 4, 3) << "\n";   // expected 8 (sum of rows 2-4, cols 1-3)
    cout << ps.rangeSum(1, 1, 2, 2) << "\n";   // expected 11 (6+3+2+0)
    cout << ps.rangeSum(1, 2, 2, 4) << "\n";   // expected 12 (3+2+1+0+1+5)

    // O(rows*cols) to build once, O(1) per query -- the inclusion-exclusion trick (add,
    // add, subtract the double-counted corner) is the key idea that extends 1D prefix
    // sums into 2 dimensions

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 7. Sliding Window vs Prefix Sum — Choosing the Right Tool

### The Why
These two patterns solve visibly different shapes of problem, but it's easy to reach for the wrong one on a new problem if you haven't explicitly separated what each one actually requires.

### Core Mechanism
- **Reach for sliding window when:** you're looking for an OPTIMAL contiguous window (longest, shortest, or "does one exist") under some constraint, and the array's elements combine MONOTONICALLY as the window grows — most commonly, all-non-negative sums, or a frequency-count style condition where "more elements can only make things more/less valid" holds consistently. This monotonicity is what makes the shrink-while-valid logic (Section 3) or jump-on-violation logic (Section 2) correct.
- **Reach for prefix sum when:** you need to answer range queries — possibly MANY of them — against an array that doesn't change (or changes rarely), especially if the array can contain negative numbers, which breaks sliding window's monotonicity assumption entirely.
- **Reach for prefix sum + hash map specifically when:** the question is "how many subarrays satisfy X" rather than "find the best/optimal subarray" — counting problems often decompose into the prefix-sum algebra from Section 5, even when a sliding window feels like it should apply at first glance.
- **A fast self-check:** if the array can contain negative numbers and the problem is about subarray sums, sliding window is very likely the wrong tool — that's usually the single fastest signal for which pattern to reach for.

### Common Pitfalls
- Attempting sliding window on a "count subarrays with sum == k" problem out of habit (it resembles Section 3's shape) without checking whether negative numbers are allowed — this is the most common misapplication of the wrong pattern between these two topics.


## 8. Phase Checkpoint, Cheat Sheet, and Answer Key

### Pattern Cheat Sheet

| Pattern | Shape | Requires | Complexity | Example |
|---|---|---|---|---|
| Fixed-size window | Slide a constant-size window, O(1) update per slide | Fixed window size given | O(n) | Max sum subarray of size k |
| Variable window (expand+jump) | `right` expands, `left` jumps on violation | Monotonic validity condition | O(n) | Longest Substring Without Repeating Characters |
| Variable window (expand+shrink) | `right` expands, `left` shrinks while still valid | Non-negative sums (monotonic growth) | O(n) | Minimum Size Subarray Sum |
| 1D prefix sum | Precompute once, O(1) per query | Static array, many range queries | O(n) build, O(1)/query | Range Sum Query - Immutable |
| Prefix sum + hash map | Track prefix sum occurrence counts | Counting subarrays, works with negatives | O(n) | Subarray Sum Equals K |
| 2D prefix sum | Inclusion-exclusion over 2 axes | Static grid, many rectangle queries | O(rows·cols) build, O(1)/query | Range Sum Query 2D - Immutable |

### Checkpoint Questions
1. Why is fixed-size sliding window O(n) instead of O(n·k)?
2. In the "longest substring without repeating characters" pattern, why does `left` jump directly to `lastSeenIndex[c] + 1` instead of incrementing one step at a time?
3. Why is a variable-size window with a nested `while` loop still O(n) overall, not O(n²)?
4. Derive the 1D prefix sum range-query formula from scratch: if `prefix[i]` is the sum of everything before index `i`, what's the sum of `nums[left..right]`?
5. Why can't sliding window correctly solve "count subarrays summing to k" when the array contains negative numbers, while prefix sum + hash map handles it fine?
6. In 2D prefix sum's build step, why is `prefix[r][c]` (the diagonal corner) subtracted, not added, when combining the cell above and the cell to the left?

### Answer Key
1. Because sliding the window only requires O(1) work per position (add the newly entering element, subtract the one leaving) — versus recomputing each window's sum from scratch, which is O(k) work repeated roughly n times.
2. Because jumping directly is still correct AND cheaper: any index between the old `left` and `lastSeenIndex[c] + 1` is guaranteed to also be fine to skip past, since the ONLY problem was the repeated character at `lastSeenIndex[c]` — there's no need to re-check each position individually one at a time when you already know exactly where the conflict is.
3. Because `left`'s total movement, summed across the ENTIRE run of the outer loop (not per individual outer iteration), is bounded by `n` — it only ever increases and can never exceed the array's length. The "nested loop" appearance is misleading; what matters is total combined work across the whole run, not the shape of the code.
4. `sum(nums[left..right]) = prefix[right+1] - prefix[left]`. `prefix[right+1]` is everything up to and including `right`; `prefix[left]` is everything strictly before `left`; subtracting the latter from the former leaves exactly the elements from `left` to `right`.
5. Sliding window's shrink logic relies on the window's sum changing monotonically as it grows — every added element makes the sum bigger (or the window "more/less valid" consistently). Negative numbers break this: adding an element could DECREASE the sum, so there's no reliable rule for when to stop shrinking or expanding. Prefix sum + hash map has no such requirement — it's pure algebra on cumulative sums and a lookup, unaffected by whether individual elements are positive or negative.
6. Because adding `prefix[r][c+1]` (sum above) and `prefix[r+1][c]` (sum to the left) each independently include the top-left diagonal region `prefix[r][c]` — so that region gets counted TWICE across the two additions. Subtracting it once corrects the double-count, leaving each region counted exactly once before the current cell's own value is added.

### Mini Project
Implement `NumMatrix` supporting `update(row, col, val)` (point update) and `sumRegion(r1, c1, r2, c2)` (range query) on a 2D grid.
1. Notice the plain 2D prefix sum from Section 6 doesn't support fast updates — a single point update would require rebuilding a large portion of the prefix table.
2. Research (or recall from the README's Advanced Trees section) how a 2D Binary Indexed Tree (Fenwick Tree) supports both point updates and range queries in O(log(rows) · log(cols)).
3. Implement it, and compare its update cost against naively rebuilding the whole prefix sum table after every point update.

This exercises: recognizing exactly where the static-array assumption behind prefix sum breaks down, and connecting back to the Fenwick Tree material from the course's foundational README — the moment "many range queries" gains "and also frequent updates" as a requirement, the right tool changes entirely.


## 9. LeetCode Practice Problems

Grouped by pattern, in rough difficulty order within each group. Hints point at the pattern's key decision, not the full solution.

### Fixed-Size Sliding Window

| # | Problem | Difficulty | Hint |
|---|---|---|---|
| 643 | Maximum Average Subarray I | Easy | Direct application of Section 1 |
| 1004 | Max Consecutive Ones III | Medium | Window size isn't literally fixed, but the CONDITION (at most k zeros) behaves like a fixed-budget window — track a zero count instead of a sum |
| 567 | Permutation in String | Medium | Fixed window size = length of the target string; compare character-frequency maps instead of a single sum |

### Variable-Size Sliding Window

| # | Problem | Difficulty | Hint |
|---|---|---|---|
| 3 | Longest Substring Without Repeating Characters | Medium | Section 2, directly |
| 209 | Minimum Size Subarray Sum | Medium | Section 3, directly |
| 76 | Minimum Window Substring | Hard | Same expand-then-shrink shape as Section 3, but "valid" means a character-frequency map fully covers the target's requirements, not a single sum threshold |
| 424 | Longest Repeating Character Replacement | Medium | Window is valid when `(window length) - (count of most frequent character) <= k` — track character counts as you expand |
| 438 | Find All Anagrams in a String | Medium | Fixed-size window (length of the target word) combined with frequency-map comparison, similar to #567 |

### 1D Prefix Sum

| # | Problem | Difficulty | Hint |
|---|---|---|---|
| 303 | Range Sum Query - Immutable | Easy | Section 4, directly |
| 724 | Find Pivot Index | Easy | The pivot is where left-side prefix sum equals right-side prefix sum — compute both directions |
| 1732 | Find the Highest Altitude | Easy | The altitude sequence itself IS a prefix sum of the gain/loss array |

### Prefix Sum + Hash Map

| # | Problem | Difficulty | Hint |
|---|---|---|---|
| 560 | Subarray Sum Equals K | Medium | Section 5, directly |
| 523 | Continuous Subarray Sum | Medium | Same prefix-sum-matching idea, but matching on remainder modulo k instead of exact value |
| 525 | Contiguous Array | Medium | Treat 0s as -1 — then "equal number of 0s and 1s" becomes "subarray sums to 0," reducing directly to the Section 5 technique |

### 2D Prefix Sum

| # | Problem | Difficulty | Hint |
|---|---|---|---|
| 304 | Range Sum Query 2D - Immutable | Medium | Section 6, directly |
| 1314 | Matrix Block Sum | Medium | Build the 2D prefix sum once, then query a clamped (edge-aware) rectangle around each cell |

### Self-Check Before Moving to `03_Hashing_and_Binary_Search`
If you can correctly decide — before writing any code — whether a new subarray/substring problem needs sliding window or prefix sum (using the negative-number test from Section 7 as a first filter), you've internalized the part of this notebook that matters most. The implementations are secondary to that judgment call.
